# IMU-Assisted Targetless Spatiotemporal Calibration Driver

Rolling-batch GCS driver for the IMU-assisted targetless radar-camera spatiotemporal calibration
described in [`docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md`](../docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md)
(read that document first -- this notebook implements its section 7 design, not a standalone
description of its own).

Runs Stages 3-5 of that plan (`branch1/calib/spatiotemporal_calibrate.py`,
`branch1/calib/doppler_lever_arm.py`, `branch1/calib/geometry_refinement.py`) over the
`ds_buildingunknown_2026-07-20` session group -- the only dataset confirmed (see
`manifests/imu_audit_2026-09-04.md`) to have per-session IMU data (`<session>_imu.csv`).

**Status disclaimer**: this notebook was written and unit-tested (see
`tests/test_doppler_and_geometry_stage_synthetic.py`) in an environment without Colab/GCS
execution access. The underlying solver code is validated against synthetic ground-truth data;
this notebook's GCS/rolling-batch orchestration has NOT been executed end-to-end. Expect to debug
real-environment issues (paths, quotas, session-count assumptions) on a first run -- start with
`MAX_BATCHES = 1` before a full sweep, per this repo's own convention for every other driver
notebook.


## Flow

1. Detect Colab vs local runtime, resolve code/work roots.
2. Configure `gcloud` auth and storage transfer tuning.
3. Sync this repo's `branch1/calib/`, `config/`, `manifests/`, `tools/` trees from GCS (a new
   prefix, not the legacy `code/canon/` other notebooks use -- see the code-sync cell for the
   one-time push command).
4. Select the session pool: `ds_buildingunknown_2026-07-20`, `capture_date >= 2026-05-01`.
5. **Pass 1** (rolling batch, fully resumable): download each session's raw bundle, run Stages 1-3
   (`process_session`), accumulate `SessionMotionData` in RAM *and* pickle-upload it to GCS
   per-session (`pass1_session_data/`, alongside a `pass1_session_diagnostics/` JSON), delete the
   raw bundle. A session already marked done in GCS -- from this run's earlier batches or a prior
   Colab runtime that disconnected -- is recovered from its pickle instead of reprocessed or
   dropped, so a disconnect only costs the session in flight, not the whole sweep.
6. **Pass 2** (single step, in-RAM): Stage 3 velocity solve, then Stage 4 Doppler-lever-arm solve.
7. **Pass 3** (rolling batch, optional): re-download each session's `depth/` only, gather Stage 5
   geometry correspondences using Stage 4's transform, delete the depth files, accumulate
   correspondences; Stage 5 solve once at the end.
8. Compare every stage's result against the static (target-based) prior, upload results + a run
   manifest to GCS.

See the plan doc's section 7 for why Pass 3 is separate from Pass 1, not folded in -- Stage 5's
data dependency (needs Stage 4's transform to decide correspondence membership, but needs depth
files on disk to do so) is a real constraint discovered while implementing
`geometry_refinement.py`, not an arbitrary design choice.


In [ ]:
from __future__ import annotations

import json
import pickle
import shutil
import subprocess
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "branch1" / "calib" / "spatiotemporal_calibrate.py").exists():
            return candidate
    return None


if IN_COLAB:
    WORK_ROOT = Path("/content/work")
    CODE_ROOT = WORK_ROOT / "code"
    STAGING_ROOT = WORK_ROOT / "staging"
    RESULTS_ROOT = WORK_ROOT / "results"
else:
    detected_root = find_repo_root(Path.cwd().resolve())
    if detected_root is None:
        raise RuntimeError(
            "Run this notebook from the RESA_mmWave repo root (or a subdirectory of it), "
            "or edit CODE_ROOT manually."
        )
    WORK_ROOT = detected_root
    CODE_ROOT = detected_root
    STAGING_ROOT = detected_root / "_calib_staging"
    RESULTS_ROOT = detected_root / "_calib_results"

for path in (WORK_ROOT, CODE_ROOT, STAGING_ROOT, RESULTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("CODE_ROOT:", CODE_ROOT)
print("STAGING_ROOT:", STAGING_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


In [ ]:
PROJECT_ID = "fluent-webbing-496616-u8"
BUCKET_NAME = "miamioh-resa-data"

# NOT the legacy code/canon/ tree other notebooks use -- this notebook targets the refactored
# RESA_mmWave repo (branch1/calib/*.py), which per the resa-pipeline-notebooks skill has never
# been pushed to GCS under code/. Following that skill's own recommendation ("prefer a new prefix
# ... over overwriting code/"), this uses a dedicated prefix. See the code-sync cell below for the
# one-time push command to populate it.
GCS_CODE_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/code_v2/RESA_mmWave/"
GCS_RAW_SESSIONS_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/raw/sessions"
GCS_RESULTS_ROOT = f"gs://{BUCKET_NAME}/CapstoneData/intermediate/spatiotemporal_calib/runs"

GCLOUD_PROCESS_COUNT = 4
GCLOUD_THREAD_COUNT = 16


def run(cmd, *, cwd: Path | None = None, check: bool = True, capture: bool = False):
    if isinstance(cmd, str):
        printable, use_shell = cmd, True
    else:
        printable, use_shell = " ".join(str(x) for x in cmd), False
    print("$", printable)
    result = subprocess.run(
        cmd, cwd=str(cwd) if cwd else None, shell=use_shell, check=check, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    return result.stdout or "" if capture else result


def gcloud_storage_rsync(src, dst, *, delete: bool = False, excludes: list[str] | None = None):
    cmd = ["gcloud", "storage", "rsync", "--recursive"]
    if delete:
        cmd.append("--delete-unmatched-destination-objects")
    for pattern in excludes or []:
        cmd.extend(["--exclude", pattern])
    cmd.extend([str(src), str(dst)])
    return run(cmd)


def gcloud_storage_cp(src, dst, *, recursive: bool = False):
    cmd = ["gcloud", "storage", "cp"]
    if recursive:
        cmd.append("--recursive")
    cmd.extend([str(src), str(dst)])
    return run(cmd)


def gcloud_storage_ls(uri: str) -> list[str]:
    if shutil.which("gcloud") is None:
        return []
    out = run(["gcloud", "storage", "ls", uri], check=False, capture=True)
    return [line.strip() for line in out.splitlines() if line.strip().startswith("gs://")]


def configure_gcloud():
    if not IN_COLAB:
        print("Local runtime: skipping Colab auth. Assumes `gcloud auth` is already configured.")
        return
    from google.colab import auth
    auth.authenticate_user()
    run(["gcloud", "config", "set", "project", PROJECT_ID])
    run(["gcloud", "config", "set", "storage/process_count", str(GCLOUD_PROCESS_COUNT)])
    run(["gcloud", "config", "set", "storage/thread_count", str(GCLOUD_THREAD_COUNT)])


configure_gcloud()


## Code sync

This notebook needs `branch1/calib/*.py`, `branch1/processing/adc_to_pointcloud_v6.py` (raw
sessions pulled from GCS have no precomputed point-cloud CSV -- Pass 1 runs this to generate one,
see `--preprocess-missing` below), `config/*.json`, `config/profile_objdet.cfg`, and
`manifests/session_catalog.csv` from **your current RESA_mmWave checkout**, not the legacy
`code/canon/` tree in GCS. Push it once (from a machine with this checkout and `gcloud` access --
this local dev machine, in this project's usual workflow) before running this notebook in Colab.

**Excludes `branch1/calib/results/`** (calibration experiment outputs -- currently ~1.5GB of
diagnostic overlay videos from unrelated prior calibration runs; this is a code sync, not a data
sync) in addition to the usual `data/`, `__pycache__/`, `.git/`:

```bash
gcloud storage rsync -r /path/to/RESA_mmWave/ gs://miamioh-resa-data/CapstoneData/code_v2/RESA_mmWave/ \
    --exclude 'data/.*' --exclude '__pycache__/.*' --exclude '\.git/.*' --exclude '_calib_.*' \
    --exclude 'branch1/calib/results/.*'
```

Re-run that command whenever you change the calibration code locally -- this notebook does not
push, only pulls.


In [ ]:
if IN_COLAB:
    gcloud_storage_rsync(
        GCS_CODE_ROOT,
        CODE_ROOT,
        excludes=[
            r"data/.*", r"__pycache__/.*", r"\.git/.*", r"_calib_.*", r"\.ipynb_checkpoints/.*",
            r"branch1/calib/results/.*",  # ~1.5GB of unrelated prior calibration experiment videos
        ],
    )
else:
    print("Local runtime: using current checkout, no code sync.")

required_repo_files = [
    CODE_ROOT / "branch1" / "calib" / "spatiotemporal_calibrate.py",
    CODE_ROOT / "branch1" / "calib" / "doppler_lever_arm.py",
    CODE_ROOT / "branch1" / "calib" / "geometry_refinement.py",
    CODE_ROOT / "branch1" / "calib" / "imu_io.py",
    CODE_ROOT / "branch1" / "calib" / "d435i_extrinsics.py",
    CODE_ROOT / "branch1" / "processing" / "adc_to_pointcloud_v6.py",  # needed by --preprocess-missing
    CODE_ROOT / "config" / "radar_camera_extrinsics.json",
    CODE_ROOT / "config" / "d435i_factory_extrinsics.json",
    CODE_ROOT / "config" / "profile_objdet.cfg",  # radar cfg for --preprocess-missing
    CODE_ROOT / "manifests" / "session_catalog.csv",
]
missing = [str(p) for p in required_repo_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing expected calibration code/config files (push your checkout to "
        f"{GCS_CODE_ROOT} first -- see the markdown cell above):\n" + "\n".join(missing)
    )
print("Calibration code layout verified.")


In [ ]:
if IN_COLAB:
    run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
    run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas", "opencv-python-headless"])
else:
    print("Local runtime: assuming numpy/scipy/pandas/opencv are already installed.")


In [ ]:
calib_dir = CODE_ROOT / "branch1" / "calib"
if str(calib_dir) not in sys.path:
    sys.path.insert(0, str(calib_dir))
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import spatiotemporal_calibrate as stc
import doppler_lever_arm as dla
import geometry_refinement as geo

print("spatiotemporal_calibrate, doppler_lever_arm, geometry_refinement imported OK")


## Session selection

Filters `manifests/session_catalog.csv` to `dataset_id == "ds_buildingunknown_2026-07-20"` (the
only group confirmed to have `_imu.csv` per `manifests/imu_audit_2026-09-04.md`) and
`capture_date >= 2026-05-01` (the rig-move cutoff -- see
`docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md` section 1). These are **raw** sessions
(`label_status == "raw"`), not curated ones, so this does not use
`tools.session_selector.select_sessions` (which defaults to requiring a non-empty
`processed_path` -- these sessions have none, by design, since this notebook consumes raw
capture data directly).


In [ ]:
import pandas as pd

RIG_MOVE_CUTOFF_DATE = "2026-05-01"
TARGET_DATASET_ID = "ds_buildingunknown_2026-07-20"

catalog_path = CODE_ROOT / "manifests" / "session_catalog.csv"
catalog = pd.read_csv(catalog_path, low_memory=False)
catalog["capture_date"] = pd.to_datetime(catalog["capture_date"], errors="coerce")

gated = catalog[
    (catalog["dataset_id"] == TARGET_DATASET_ID)
    & (catalog["capture_date"] >= pd.Timestamp(RIG_MOVE_CUTOFF_DATE))
].sort_values("session_id").reset_index(drop=True)

print(f"Catalog rows total: {len(catalog)}")
print(f"Gated sessions ({TARGET_DATASET_ID}, capture_date >= {RIG_MOVE_CUTOFF_DATE}): {len(gated)}")


def session_raw_gcs_uri(row: pd.Series) -> str:
    """Prefer the catalog's own raw_path (manifest is authoritative per
    resa-data-contracts skill); fall back to the confirmed GCS layout
    convention if that column is empty for this row."""
    raw_path = str(row.get("raw_path", "") or "").strip()
    if raw_path:
        return raw_path.rstrip("/") + "/"
    return f"{GCS_RAW_SESSIONS_ROOT}/{row['dataset_id']}/{row['session_id']}/"


session_ids = gated["session_id"].tolist()
session_uris = {row["session_id"]: session_raw_gcs_uri(row) for _, row in gated.iterrows()}
print("Example session URI:", session_uris[session_ids[0]] if session_ids else "(none gated)")


## Run configuration

Start with `MAX_BATCHES = 1` for a smoke run against a couple of sessions before a full sweep,
per this repo's convention for every rolling-batch notebook. `--time-mode solve` is intentionally
not exposed here as an option -- see `docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md` section 5.1
for why it's currently non-functional; this notebook always uses `time_mode="fixed"`.


In [ ]:
BATCH_SIZE = 20
MAX_BATCHES = 1  # None for a full run; keep at 1 for the first smoke test
OMEGA_SOURCE = "auto"  # "auto" | "rgbd" | "imu" -- see spatiotemporal_calibrate.py --omega-source help
CALIBRATION_STAGE = "doppler"  # "velocity" (Stage 3 only) | "doppler" (Stage 3 -> Stage 4)
# Manual override for the radar<->camera time offset, in milliseconds, signed.
# Convention: camera_time = radar_time + offset (see gather_pairs' query_us computation).
# This rig's radar chain is known (from the prior rig generation's identical electronics)
# to under-report timestamps by ~250ms, so camera time must be pushed FORWARD to
# compensate -- i.e. the expected correct sign is POSITIVE (+250.0), NOT the -125ms the
# algorithmic estimator produced on the last run. Set None to use the existing
# algorithmic estimation (choose_temporal_initialization); set a fixed value to override.
FIXED_TIME_OFFSET_MS = None  # e.g. 250.0 to test the known hardware latency
REFINE_GEOMETRY = True  # Stage 5, only meaningful when CALIBRATION_STAGE == "doppler"

FORCE_REEXTRACT_SESSIONS = False  # set True to reprocess sessions with existing Pass-1 diagnostics

STATIC_PRIOR_PATH = CODE_ROOT / "config" / "radar_camera_extrinsics.json"
# Both prefixes are deliberately NOT under RUN_ID (unlike GCS_RESULTS_ROOT/{RUN_ID}/ below) --
# they're a cross-run cache keyed only by session_id, so a fresh RUN_ID (e.g. after a Colab
# disconnect) still finds every session a previous run already extracted. See run_pass1()'s
# docstring for how this makes Pass 1 fully resumable.
PASS1_DIAGNOSTICS_GCS_PREFIX = f"{GCS_RESULTS_ROOT}/pass1_session_diagnostics"
PASS1_DATA_GCS_PREFIX = f"{GCS_RESULTS_ROOT}/pass1_session_data"

RUN_ID = f"imu_spatiotemporal_calib_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
RUN_RESULTS_DIR = RESULTS_ROOT / RUN_ID
RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_ID:", RUN_ID)

# Build a fully-defaulted args Namespace the same way the CLI would, then
# override just what this notebook's config cares about -- avoids having to
# hand-construct spatiotemporal_calibrate.py's ~50-field argparse surface.
calib_args = stc.build_arg_parser().parse_args([])
calib_args.omega_source = OMEGA_SOURCE
calib_args.calibration_stage = CALIBRATION_STAGE
calib_args.fixed_time_offset_ms = FIXED_TIME_OFFSET_MS
# NOTE: this is NOT the same flag as the REFINE_GEOMETRY toggle above, despite the similar name.
# calib_args.refine_geometry controls whether process_session() builds a DepthLookup during Pass
# 1 -- forced False here because Pass 1 deletes each session's raw files (including depth/) right
# after processing it, and a DepthLookup's closure would try to read from those deleted files if
# built. REFINE_GEOMETRY (the plain Python variable) is what actually gates whether Pass 3 /
# Stage 5 runs at all, further down -- Pass 3 builds its OWN fresh DepthLookup per session, after
# re-downloading just that session's depth files. It also means every pickled SessionMotionData
# (see upload_pass1_data in the next cell) never references files on local disk.
calib_args.refine_geometry = False
# REQUIRED: sessions pulled raw from GCS have no precomputed point-cloud CSV
# (that's a downstream pipeline artifact, not part of the raw capture bundle
# -- see resa-data-contracts skill). process_session() raises
# FileNotFoundError otherwise (spatiotemporal_calibrate.py::_preprocess_if_needed).
calib_args.preprocess_missing = True
calib_args.adc_script = CODE_ROOT / "branch1" / "processing" / "adc_to_pointcloud_v6.py"
calib_args.adc_cfg = CODE_ROOT / "config" / "profile_objdet.cfg"
calib_args.static_prior = STATIC_PRIOR_PATH
calib_args.output_dir = RUN_RESULTS_DIR
print(f"omega_source={calib_args.omega_source} calibration_stage={calib_args.calibration_stage} "
      f"fixed_time_offset_ms={calib_args.fixed_time_offset_ms} "
      f"preprocess_missing={calib_args.preprocess_missing}")


In [ ]:
def batched(items: list, size: int):
    for i in range(0, len(items), size):
        yield items[i : i + size]


def pass1_diagnostics_gcs_uri(session_id: str) -> str:
    return f"{PASS1_DIAGNOSTICS_GCS_PREFIX}/{session_id}.json"


def pass1_data_gcs_uri(session_id: str) -> str:
    return f"{PASS1_DATA_GCS_PREFIX}/{session_id}.pkl"


def pass1_already_done(session_id: str) -> bool:
    """True once this session has a diagnostics JSON in GCS from ANY prior
    run (this run's own earlier batches, or an earlier Colab runtime that
    disconnected) -- diagnostics are uploaded for every processed session,
    usable or not, right after process_session() returns, so this is the
    authoritative "already attempted" check independent of whether that
    attempt found the session usable."""
    if FORCE_REEXTRACT_SESSIONS:
        return False
    return len(gcloud_storage_ls(pass1_diagnostics_gcs_uri(session_id))) > 0


def download_session_raw(session_id: str) -> Path:
    dest = STAGING_ROOT / session_id
    dest.mkdir(parents=True, exist_ok=True)
    gcloud_storage_rsync(session_uris[session_id], dest)
    return dest


def clear_local_session(session_id: str) -> None:
    dest = STAGING_ROOT / session_id
    if dest.exists():
        shutil.rmtree(dest, ignore_errors=True)


def upload_pass1_diagnostics(session_id: str, info: dict[str, Any]) -> None:
    local_path = RUN_RESULTS_DIR / "pass1_diagnostics" / f"{session_id}.json"
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(json.dumps(info, indent=2, default=str))
    gcloud_storage_cp(local_path, pass1_diagnostics_gcs_uri(session_id))


def upload_pass1_data(session_id: str, data: Any) -> None:
    """Persist a usable session's SessionMotionData to GCS right after it's
    computed, so a Colab disconnect mid-sweep loses at most the session
    in flight, not every session processed before it. Without this, only
    the diagnostics JSON (a small metrics dict, not the actual
    velocity/pose/IMU arrays Pass 2/3 need) would survive a disconnect, so a
    resumed run would either have to reprocess every "already done" session
    from raw again, or -- as the notebook used to do -- silently treat them
    as done-and-unrecoverable, which fails outright once every session is
    marked done. depth_lookup is always None here (calib_args.refine_geometry
    is forced False for Pass 1), so this pickle never references files on
    local disk; angular_velocity_at is a picklable callable
    (spatiotemporal_calibrate._IMUAngularVelocityAt, or a bound method on
    the picklable CameraTrajectory), not a bare lambda closure, specifically
    so this round-trips correctly."""
    local_path = RUN_RESULTS_DIR / "pass1_data" / f"{session_id}.pkl"
    local_path.parent.mkdir(parents=True, exist_ok=True)
    with local_path.open("wb") as fh:
        pickle.dump(data, fh, protocol=pickle.HIGHEST_PROTOCOL)
    gcloud_storage_cp(local_path, pass1_data_gcs_uri(session_id))


def recover_pass1_session(session_id: str) -> tuple[Any, dict[str, Any]]:
    """Reload one already-done session's diagnostics from GCS, and its
    SessionMotionData too if it was usable last time (no data pickle in GCS
    means it was processed and found unusable -- e.g. RGB-D odometry
    failure -- so there's nothing to recover, same as a fresh skip)."""
    diag_local = STAGING_ROOT / "_pass1_diag_cache" / f"{session_id}.json"
    diag_local.parent.mkdir(parents=True, exist_ok=True)
    gcloud_storage_cp(pass1_diagnostics_gcs_uri(session_id), diag_local)
    info = json.loads(diag_local.read_text())
    diag_local.unlink(missing_ok=True)

    if len(gcloud_storage_ls(pass1_data_gcs_uri(session_id))) == 0:
        return None, info

    data_local = STAGING_ROOT / "_pass1_data_cache" / f"{session_id}.pkl"
    data_local.parent.mkdir(parents=True, exist_ok=True)
    gcloud_storage_cp(pass1_data_gcs_uri(session_id), data_local)
    with data_local.open("rb") as fh:
        data = pickle.load(fh)
    data_local.unlink(missing_ok=True)
    return data, info


def run_pass1() -> tuple[list, dict[str, Any]]:
    """Stages 1-3 extraction, rolling batch. Returns (usable_data,
    per-session diagnostics dict). depth_lookup is deliberately never built
    here (calib_args.refine_geometry is forced False) -- see markdown cell
    above the Pass 3 section for why.

    Fully resumable: every usable session's SessionMotionData is pickled and
    uploaded to GCS (upload_pass1_data) immediately after it's computed,
    alongside its diagnostics JSON. Every already-done session (this run's
    earlier batches, or a prior Colab runtime that disconnected) is recovered
    from GCS via recover_pass1_session() instead of being reprocessed OR
    silently dropped -- the latter is what this notebook used to do, which
    meant a disconnect partway through a long sweep discarded every session
    processed before it, and if every gated session ended up already-done,
    Pass 2 saw zero usable_data and the whole run failed outright."""
    usable_data: list = []
    diagnostics: dict[str, Any] = {}
    already_done = [sid for sid in session_ids if pass1_already_done(sid)]
    todo = [sid for sid in session_ids if sid not in set(already_done)]
    print(f"Pass 1: {len(already_done)}/{len(session_ids)} sessions already done -- recovering "
          f"from GCS; {len(todo)} remaining need extraction "
          f"(FORCE_REEXTRACT_SESSIONS={FORCE_REEXTRACT_SESSIONS})")

    for session_id in already_done:
        data, info = recover_pass1_session(session_id)
        diagnostics[session_id] = info
        if data is not None:
            usable_data.append(data)
            print(f"  {session_id}: recovered from GCS (usable)")
        else:
            print(f"  {session_id}: recovered from GCS (previously unusable -- {info.get('error')})")

    for batch_idx, batch in enumerate(batched(todo, BATCH_SIZE), start=1):
        if MAX_BATCHES is not None and batch_idx > MAX_BATCHES:
            print(f"MAX_BATCHES={MAX_BATCHES} reached, stopping Pass 1 early.")
            break
        print(f"--- Pass 1 batch {batch_idx}: {len(batch)} sessions ---")
        for session_id in batch:
            session_dir = download_session_raw(session_id)
            data, info = stc.process_session(session_dir, calib_args)
            diagnostics[session_id] = info
            upload_pass1_diagnostics(session_id, info)
            if data is not None:
                upload_pass1_data(session_id, data)
                usable_data.append(data)
                agreement = info.get("omega_agreement", {}).get("median_abs_diff_radps")
                print(f"  {session_id}: usable omega_source={info.get('omega_source_used')} "
                      f"omega_agree_radps={agreement}")
            else:
                print(f"  {session_id}: skipped -- {info.get('error')}")
            clear_local_session(session_id)
        run(["df", "-h", "/content"] if IN_COLAB else ["df", "-h", str(STAGING_ROOT)], check=False)

    return usable_data, diagnostics


In [ ]:
usable_data, pass1_diagnostics = run_pass1()
print(f"Pass 1 complete: {len(usable_data)} usable sessions out of {len(session_ids)} gated")
if not usable_data:
    raise SystemExit(
        "No usable sessions after Pass 1 -- check pass1_diagnostics for per-session error reasons "
        "before proceeding (common causes: MAX_BATCHES too low, missing raw files, RGB-D odometry "
        "failures -- see each session's 'error' field)."
    )


## Pass 2: Stage 3 (velocity) + Stage 4 (Doppler lever-arm) solve

Single step, in-RAM -- no GCS/disk I/O beyond writing the result JSONs. Mirrors
`spatiotemporal_calibrate.py`'s own `main()` control flow exactly (this notebook calls the same
functions `main()` calls, rather than shelling out to the CLI, so it can hold `usable_data` in
memory across Pass 1/2/3 instead of round-tripping through argv/files).


In [ ]:
static_prior = stc.load_static_prior(STATIC_PRIOR_PATH)
print(f"Static prior loaded: {static_prior.loaded} (from {STATIC_PRIOR_PATH})")

stage3_result = stc.solve_spatiotemporal(usable_data, static_prior, calib_args)
print(f"Stage 3 (velocity): paired={stage3_result['paired_motion_samples']} "
      f"residual_rms_mps={stage3_result['residual_rms_mps']:.4f} "
      f"fallback={stage3_result['fallback']}")

(RUN_RESULTS_DIR / "stage3_velocity_radar_camera_extrinsics.json").write_text(json.dumps(stage3_result, indent=2))

final_result = stage3_result
stage4_result = None
if CALIBRATION_STAGE == "doppler":
    stage4_result = dla.solve_doppler_lever_arm(usable_data, static_prior, stage3_result, calib_args)
    print(f"Stage 4 (Doppler): pooled={stage4_result['n_detections_pooled']} "
          f"residual_rms_mps={stage4_result['residual_rms_mps']:.4f} "
          f"fallback={stage4_result['fallback']}")
    (RUN_RESULTS_DIR / "stage4_doppler_radar_camera_extrinsics.json").write_text(json.dumps(stage4_result, indent=2))
    final_result = stage4_result

gcloud_storage_cp(RUN_RESULTS_DIR / "stage3_velocity_radar_camera_extrinsics.json", f"{GCS_RESULTS_ROOT}/{RUN_ID}/")
if stage4_result is not None:
    gcloud_storage_cp(RUN_RESULTS_DIR / "stage4_doppler_radar_camera_extrinsics.json", f"{GCS_RESULTS_ROOT}/{RUN_ID}/")


## Pass 3: Stage 5 geometry refinement (optional, rolling batch)

Only runs if `REFINE_GEOMETRY = True`. This is a **separate rolling-batch pass**, not folded into
Pass 1, because of a real data-flow constraint: `gather_geometry_correspondences` needs Stage 4's
transform to decide which detections are valid correspondences, but Stage 4's transform is only
known *after* every session's Doppler data has already been pooled in Pass 2 -- and Pass 1 already
deleted the raw session bundles (including `depth/*.npy`) to free disk. So this re-downloads just
the small `depth/` folder + `meta_data.json` per session (not the full raw bundle, which Pass 1
already consumed once), gathers correspondences using Stage 4's now-known transform, deletes the
depth files again, and accumulates the much smaller `GeometryCorrespondence` objects (a point +
weight + one cached depth image, not a whole session's worth of files) across batches. See
`geometry_refinement.py`'s `refine_with_geometry` docstring and
`docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md` section 7 for the full rationale.


In [ ]:
def download_session_depth_only(session_id: str) -> Path:
    dest = STAGING_ROOT / session_id
    dest.mkdir(parents=True, exist_ok=True)
    base_uri = session_uris[session_id].rstrip("/")
    gcloud_storage_rsync(f"{base_uri}/depth", dest / "depth")
    gcloud_storage_cp(f"{base_uri}/meta_data.json", dest / "meta_data.json")
    # build_depth_lookup also needs the color/depth timestamp CSVs to
    # compute the color-frame -> depth-frame index mapping (no video
    # decoding needed, see spatiotemporal_calibrate.py::build_depth_lookup)
    gcloud_storage_cp(f"{base_uri}/{session_id}_color_timestamps.csv", dest / f"{session_id}_color_timestamps.csv")
    gcloud_storage_cp(f"{base_uri}/{session_id}_depth_timestamps.csv", dest / f"{session_id}_depth_timestamps.csv")
    return dest


def run_pass3(stage4_result: dict[str, Any]) -> list:
    """Rolling-batch geometry-correspondence gathering. Returns the
    accumulated list of GeometryCorrespondence objects (safe to hold across
    batches -- each one caches its own depth image array, no longer
    references the session's files on disk once gathered)."""
    all_correspondences: list = []
    data_by_session_name = {d.session_name: d for d in usable_data}

    for batch_idx, batch in enumerate(batched(list(data_by_session_name.keys()), BATCH_SIZE), start=1):
        if MAX_BATCHES is not None and batch_idx > MAX_BATCHES:
            print(f"MAX_BATCHES={MAX_BATCHES} reached, stopping Pass 3 early.")
            break
        print(f"--- Pass 3 batch {batch_idx}: {len(batch)} sessions ---")
        batch_sessions = []
        for session_id in batch:
            session_dir = download_session_depth_only(session_id)
            data = data_by_session_name[session_id]
            data.depth_lookup = stc.build_depth_lookup(session_dir)
            if data.depth_lookup is not None:
                batch_sessions.append(data)

        warm_r = stage4_result["R_radar_to_camera"]
        warm_t = stage4_result["t_radar_to_camera"]
        import numpy as np
        batch_correspondences = geo.gather_geometry_correspondences(
            batch_sessions, np.asarray(warm_r), np.asarray(warm_t),
            max_per_session=calib_args.geometry_max_detections_per_session,
        )
        print(f"  gathered {len(batch_correspondences)} correspondences from this batch")
        all_correspondences.extend(batch_correspondences)

        for session_id in batch:
            data_by_session_name[session_id].depth_lookup = None  # invalidate: files about to be deleted
            clear_local_session(session_id)
        run(["df", "-h", "/content"] if IN_COLAB else ["df", "-h", str(STAGING_ROOT)], check=False)

    return all_correspondences


In [ ]:
stage5_result = None
if REFINE_GEOMETRY and CALIBRATION_STAGE == "doppler" and stage4_result is not None:
    correspondences = run_pass3(stage4_result)
    print(f"Pass 3 complete: {len(correspondences)} correspondences pooled")
    stage5_result = geo.solve_geometry_refinement_from_correspondences(correspondences, stage4_result, calib_args)
    print(f"Stage 5 (geometry): correspondences={stage5_result['n_correspondences']} "
          f"residual_rms_m={stage5_result['residual_rms_m']:.4f} fallback={stage5_result['fallback']}")
    (RUN_RESULTS_DIR / "stage5_geometry_radar_camera_extrinsics.json").write_text(json.dumps(stage5_result, indent=2))
    gcloud_storage_cp(RUN_RESULTS_DIR / "stage5_geometry_radar_camera_extrinsics.json", f"{GCS_RESULTS_ROOT}/{RUN_ID}/")
    final_result = stage5_result
else:
    print("Skipping Pass 3 / Stage 5 (REFINE_GEOMETRY is False, or Stage 4 did not run).")


## Final comparison and run manifest

Compares every stage's result against the static (target-based) calibration using the metrics
from the reference document's section 16: `e_R = acos((tr(R_ref^T R_est) - 1) / 2)` and
`e_t = ||t_ref - t_est||`.


In [ ]:
import numpy as np


def rotation_angle_error_deg(r_ref: np.ndarray, r_est: np.ndarray) -> float:
    cos_angle = np.clip((np.trace(r_ref.T @ r_est) - 1) / 2, -1, 1)
    return float(np.degrees(np.arccos(cos_angle)))


def translation_norm_error_m(t_ref: np.ndarray, t_est: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(t_ref) - np.asarray(t_est)))


r_ref = static_prior.rotation
t_ref = static_prior.translation_m

comparison_rows = []
for stage_name, result in [
    ("stage3_velocity", stage3_result),
    ("stage4_doppler", stage4_result),
    ("stage5_geometry", stage5_result),
]:
    if result is None:
        continue
    r_est = np.asarray(result["R_radar_to_camera"])
    t_est = np.asarray(result["t_radar_to_camera"])
    comparison_rows.append({
        "stage": stage_name,
        "e_R_deg": rotation_angle_error_deg(r_ref, r_est),
        "e_t_m": translation_norm_error_m(t_ref, t_est),
        "fallback_triggered": result.get("fallback", {}).get("use_warm_start")
            or result.get("fallback", {}).get("use_doppler_input", False),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

run_manifest = {
    "run_id": RUN_ID,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_gated_sessions": len(session_ids),
    "n_usable_sessions": len(usable_data),
    "calibration_stage": CALIBRATION_STAGE,
    "omega_source": OMEGA_SOURCE,
    "refine_geometry": REFINE_GEOMETRY,
    "static_prior_path": str(STATIC_PRIOR_PATH),
    "final_method": final_result["method"],
    "comparison_vs_static_prior": comparison_rows,
}
(RUN_RESULTS_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2, default=str))
(RUN_RESULTS_DIR / "final_radar_camera_extrinsics.json").write_text(json.dumps(final_result, indent=2, default=str))
gcloud_storage_cp(RUN_RESULTS_DIR / "run_manifest.json", f"{GCS_RESULTS_ROOT}/{RUN_ID}/")
gcloud_storage_cp(RUN_RESULTS_DIR / "final_radar_camera_extrinsics.json", f"{GCS_RESULTS_ROOT}/{RUN_ID}/")
print(f"Run manifest and final result uploaded to {GCS_RESULTS_ROOT}/{RUN_ID}/")


## Known limitations and next steps

- **This notebook has not been executed end-to-end** (written without Colab/GCS access) -- the
  underlying solver code is validated against synthetic ground truth
  (`tests/test_doppler_and_geometry_stage_synthetic.py`), but this notebook's rolling-batch
  orchestration, GCS paths, and session-count assumptions are not. Start with `MAX_BATCHES = 1`.
- **Pass 1 is fully resumable, Pass 3 is not (yet).** Pass 1 pickles and uploads each usable
  session's `SessionMotionData` to `pass1_session_data/` as soon as it's computed, so a Colab
  disconnect only costs the session in flight -- re-running the notebook recovers every earlier
  session from GCS instead of reprocessing raw bundles. Pass 3's `GeometryCorrespondence` list is
  still only accumulated in RAM across its own batches, so a disconnect *during* Pass 3 loses that
  pass's progress (though it only has to redo the much cheaper `depth/`-only re-download, not
  Pass 1's full raw bundle, since Pass 1's results are unaffected).
- IMU coverage is currently limited to `ds_buildingunknown_2026-07-20` (~644 sessions) -- see
  `manifests/imu_audit_2026-09-04.md`. `OMEGA_SOURCE = "auto"` will silently fall back to
  RGB-D-differenced omega for any session lacking usable IMU data; set `OMEGA_SOURCE = "imu"` for
  strict exclusion instead (see `spatiotemporal_calibrate.py --omega-source` help).
- `--time-mode solve` (jointly solving the radar-camera time offset) is not exposed here because
  it is currently non-functional -- see `docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md` section 5.1
  for the full root-cause writeup (a scipy finite-difference / integer-microsecond-rounding
  interaction, pre-existing in Stage 3, not introduced by Stage 4).
- Stage 5 uses the *simple* projective depth residual, not point-to-plane (see
  `geometry_refinement.py`'s module docstring) -- point-to-plane is a documented future upgrade.
- Anti-circularity: every stage's correspondences come from Stage 2's RANSAC-accepted static
  radar returns only, never a Branch 1 directness/ghost classifier (this repo does not yet have
  one calibrated against this rig generation) -- see the reference document's section 8.1.
- A dedicated multi-axis excited calibration sequence (yaw+pitch+roll+translation with reversals)
  would likely outperform pooling ordinary driving sessions -- see the reference document's
  section 10.5 and `docs/IMU_SPATIOTEMPORAL_CALIBRATION_PLAN.md` open item 4.
